## Bibliotecas

In [1]:
import os

import torch
from torch import Tensor
import torch.nn.functional as F
from typing import Tuple
from torchmetrics.classification import BinaryAccuracy
import time 

from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix,accuracy_score, precision_score, f1_score, recall_score, matthews_corrcoef

import matplotlib.pyplot as plt

import numpy as np

from dataclasses import dataclass

import pandas as pd
import matplotlib.pyplot as plt
from torch.autograd import Variable
import wandb
import copy
from dataclasses import asdict

from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import ConfusionMatrixDisplay

import timm_3d

#Modelos
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

/home/labian-2/Desktop/Crateus/odonto_forense/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Carregamento dos dados

In [2]:
# Redes utilizadas para extração de carcaterísticas
resnets =['resnet10t.c3_in1k', 'resnet18.a1_in1k', 'resnet50.a1_in1k']
densenets = ['densenet169','densenet201']
efficient_net = ['efficientnet_b0.ra_in1k']
vgg = ['vgg16']
mobile_net=['mobilenetv2_100.ra_in1k']
convnexts = ['convnext_base']

# Nome das pastas dos respectivos modelos
model_map = {
    resnets[0]: "Resnet10",
    resnets[1]: "Resnet18",
    resnets[2]: "Resnet50",
    densenets[0]: "Densenet169",
    densenets[1]: "Densenet201",
    efficient_net[0]: "EfficientNetB0",
    vgg[0]: "VGG16",
    mobile_net[0]: "MobileNetV2_100",
    convnexts[0]: "ConvNext_Base"
}

In [3]:
extractor_model = resnets[1]
fold_name = model_map[extractor_model]

# Recuperando os mapas de características extraídos pelo modelo
#load_features= torch.load(f'/home/labian-2/Desktop/Crateus/odonto_forense/logs binary/{fold_name}/model_maps/model_maps.pt', weights_only=False)
load_features= torch.load(f'/home/labian-2/Desktop/Crateus/odonto_forense/testeee.pt', weights_only=False)

# Extrai os tensores e labels das features 
feature_tensors = [item["feature"] for item in load_features]
labels_tensors = [item["label"] for item in load_features]

features_tensor = torch.stack(feature_tensors)
labels_tensor = torch.tensor(labels_tensors)

# Converte de tensor para numpy - Serão meus dados
X = features_tensor.numpy()
y = labels_tensor.numpy()

print(fold_name)
print(X.shape, y.shape) 


Resnet18
(108, 2048) (108,)


## MAIS BÁSICO IMPOSSÍVEL

Separação entre treio e teste

In [4]:
from sklearn.model_selection import train_test_split

test_frac = 0.3

train_x, test_x, train_y, test_y = train_test_split(X,y , random_state=104,  test_size=test_frac,  shuffle=True, stratify=y)

print(f"Dados para treinamento:")
print(train_x)
print(f"Labels para treinamento:")
print(train_y)

print("-" * 90)
print(f"Dados para teste:")
print(test_x)
print("-" * 90)
print(f"Labels para teste:")
print(test_y)

print("-" * 90)
print(f"Treino")
print(f"Quantidade de homens: {np.sum(train_y == 0)}, Quantidade de mulheres: {np.sum(train_y == 1)}")
print(f"\nTeste")
print(f"Quantidade de homens: {np.sum(test_y == 0)}, Quantidade de mulheres: {np.sum(test_y == 1)}")

Dados para treinamento:
[[7.6286107e-02 1.8203865e-04 2.6787037e-01 ... 5.9138650e-01
  1.0662816e+01 0.0000000e+00]
 [6.5427832e-02 1.3147861e-03 1.6934451e-01 ... 5.5621898e-01
  5.5742407e+00 0.0000000e+00]
 [4.7131568e-02 2.4783742e-04 1.7501800e-01 ... 5.3055322e-01
  6.6082506e+00 0.0000000e+00]
 ...
 [1.8064229e-01 5.4841787e-03 3.2978058e-01 ... 1.0342753e+00
  1.2162716e+01 5.4709835e-04]
 [2.3548278e-01 0.0000000e+00 9.3852758e-01 ... 1.2927871e+00
  2.5083900e+01 1.1181441e-03]
 [1.3284640e-01 7.2604336e-04 5.8220714e-01 ... 9.0513921e-01
  1.2013731e+01 0.0000000e+00]]
Labels para treinamento:
[1 0 1 0 1 0 0 0 1 1 0 1 1 1 1 1 1 0 0 0 0 0 1 0 0 1 1 1 0 1 0 1 0 1 1 0 0
 1 0 0 1 0 0 0 1 1 0 1 1 0 0 0 1 0 1 1 0 0 0 0 1 1 0 1 0 1 0 1 1 0 0 0 1 1
 1]
------------------------------------------------------------------------------------------
Dados para teste:
[[5.1585369e-02 2.9123256e-03 1.6610298e-01 ... 3.3623782e-01
  7.0293045e+00 0.0000000e+00]
 [9.1789886e-02 1.9664629e-03 9

In [5]:
test_x.shape

(33, 2048)

In [6]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(train_x)
X_test_scaled = scaler.transform(test_x)

pca = PCA(n_components=0.95)

X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print("Shape reduzido do treino:", X_train_pca.shape) 
print("Shape reduzido do teste:", X_test_pca.shape) 


Shape reduzido do treino: (75, 30)
Shape reduzido do teste: (33, 30)


In [7]:
def SVM_classifier(X_train, X_test, y_train, y_test):
    conf_mat = np.zeros((2,2))

    svm_model = SVC(kernel='rbf')

    train_begin = time.time()
    svm_model.fit(X_train, y_train)
    train_end = time.time()

    train_time = train_end - train_begin
    print(f'Train time: {train_time}')

    test_begin = time.time()
    y_pred = svm_model.predict(X_test)
    test_end = time.time()

    test_time= test_end - test_begin
    print(f'Test time: {test_time}')

    test_acc = accuracy_score(y_test, y_pred)
    print(f'Acurácia do teste: {test_acc}')

    cm = confusion_matrix(y_test, y_pred)
    conf_mat += cm

    print('------- Matriz de confusão --------')
    print(cm, end='\n')

def KNN_classifier(X_train, X_test, y_train, y_test):
    conf_mat = np.zeros((2,2))

    knn_model = KNeighborsClassifier(n_neighbors=5)

    train_begin = time.time()
    knn_model.fit(X_train, y_train)
    train_end = time.time()

    train_time = train_end - train_begin
    print(f'Train time: {train_time}')

    test_begin = time.time()
    y_pred = knn_model.predict(X_test)
    test_end = time.time()

    test_time= test_end - test_begin
    print(f'Test time: {test_time}')

    test_acc = accuracy_score(y_test, y_pred)
    print(f'Acurácia do teste: {test_acc}')

    cm = confusion_matrix(y_test, y_pred)
    conf_mat += cm

    print('------- Matriz de confusão --------')
    print(cm, end='\n')

def MLP_classifier(X_train, X_test, y_train, y_test):
    conf_mat = np.zeros((2,2))

    mlp_model = MLPClassifier(random_state=1, max_iter = 300,verbose=1)

    train_begin = time.time()
    mlp_model.fit(X_train, y_train)
    train_end = time.time()

    train_time = train_end - train_begin
    print(f'Train time: {train_time}')

    test_begin = time.time()
    y_pred = mlp_model.predict(X_test)
    test_end = time.time()

    test_time= test_end - test_begin
    print(f'Test time: {test_time}')

    test_acc = accuracy_score(y_test, y_pred)
    print(f'Acurácia do teste: {test_acc}')

    cm = confusion_matrix(y_test, y_pred)
    conf_mat += cm

    print('------- Matriz de confusão --------')
    print(cm, end='\n')

def RF_classifier(X_train, X_test, y_train, y_test):
    conf_mat = np.zeros((2,2))

    rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

    train_begin = time.time()
    rf_model.fit(X_train, y_train)
    train_end = time.time()

    train_time = train_end - train_begin
    print(f'Train time: {train_time}')

    test_begin = time.time()
    y_pred = rf_model.predict(X_test)
    test_end = time.time()

    test_time= test_end - test_begin
    print(f'Test time: {test_time}')

    test_acc = accuracy_score(y_test, y_pred)
    print(f'Acurácia do teste: {test_acc}')

    cm = confusion_matrix(y_test, y_pred)
    conf_mat += cm

    print('------- Matriz de confusão --------')
    print(cm, end='\n')


In [8]:
print("============ SVM Resultados sem PCA ============")
sem_pca = SVM_classifier(train_x, test_x, train_y, test_y)

============ SVM Resultados sem PCA ============
Train time: 0.0037412643432617188
Test time: 0.0018830299377441406
Acurácia do teste: 0.6060606060606061
------- Matriz de confusão --------
[[10  6]
 [ 7 10]]


In [8]:
print("============ SVM Resultados sem PCA ============")
sem_pca = SVM_classifier(train_x, test_x, train_y, test_y)
print("============ SVM Resultados com PCA ============")
com_pca = SVM_classifier(X_train_pca, X_test_pca, train_y, test_y)

============ SVM Resultados sem PCA ============
Train time: 0.04427790641784668
Test time: 0.0010099411010742188
Acurácia do teste: 0.9090909090909091
------- Matriz de confusão --------
[[14  2]
 [ 1 16]]
============ SVM Resultados com PCA ============
Train time: 0.0006594657897949219
Test time: 0.0002110004425048828
Acurácia do teste: 0.9393939393939394
------- Matriz de confusão --------
[[15  1]
 [ 1 16]]


In [9]:
print("============ KNN Resultados sem PCA ============")
sem_pca = KNN_classifier(train_x, test_x, train_y, test_y)

============ KNN Resultados sem PCA ============
Train time: 0.0008749961853027344
Test time: 0.06857991218566895
Acurácia do teste: 0.6060606060606061
------- Matriz de confusão --------
[[ 8  8]
 [ 5 12]]


In [9]:
print("============ KNN Resultados sem PCA ============")
sem_pca = KNN_classifier(train_x, test_x, train_y, test_y)
print("============ KNN Resultados com PCA ============")
com_pca = KNN_classifier(X_train_pca, X_test_pca, train_y, test_y)

============ KNN Resultados sem PCA ============
Train time: 0.0005943775177001953
Test time: 0.18021273612976074
Acurácia do teste: 0.9393939393939394
------- Matriz de confusão --------
[[15  1]
 [ 1 16]]
============ KNN Resultados com PCA ============
Train time: 0.0003349781036376953
Test time: 0.0005519390106201172
Acurácia do teste: 0.9393939393939394
------- Matriz de confusão --------
[[15  1]
 [ 1 16]]


In [10]:
print("============ RF Resultados sem PCA ============")
sem_pca = RF_classifier(train_x, test_x, train_y, test_y)
print("============ RF Resultados com PCA ============")
com_pca = RF_classifier(X_train_pca, X_test_pca, train_y, test_y)

============ RF Resultados sem PCA ============
Train time: 0.09386706352233887
Test time: 0.003668069839477539
Acurácia do teste: 0.9393939393939394
------- Matriz de confusão --------
[[15  1]
 [ 1 16]]
============ RF Resultados com PCA ============
Train time: 0.07294368743896484
Test time: 0.0037622451782226562
Acurácia do teste: 0.9393939393939394
------- Matriz de confusão --------
[[15  1]
 [ 1 16]]


In [11]:
print("============ MLP Resultados sem PCA ============")
sem_pca = MLP_classifier(train_x, test_x, train_y, test_y)
print("============ MLP Resultados com PCA ============")
com_pca = MLP_classifier(X_train_pca, X_test_pca, train_y, test_y)

============ MLP Resultados sem PCA ============
Iteration 1, loss = 0.72766649
Iteration 2, loss = 0.55180442
Iteration 3, loss = 0.46000941
Iteration 4, loss = 0.39428769
Iteration 5, loss = 0.34230793
Iteration 6, loss = 0.30037215
Iteration 7, loss = 0.26629345
Iteration 8, loss = 0.23867989
Iteration 9, loss = 0.21628733
Iteration 10, loss = 0.19842600
Iteration 11, loss = 0.18400570
Iteration 12, loss = 0.17232429
Iteration 13, loss = 0.16274942
Iteration 14, loss = 0.15466133
Iteration 15, loss = 0.14758449
Iteration 16, loss = 0.14124726
Iteration 17, loss = 0.13549633
Iteration 18, loss = 0.13022485
Iteration 19, loss = 0.12536622
Iteration 20, loss = 0.12091425
Iteration 21, loss = 0.11680133
Iteration 22, loss = 0.11296783
Iteration 23, loss = 0.10933519
Iteration 24, loss = 0.10583582
Iteration 25, loss = 0.10242745
Iteration 26, loss = 0.09907359
Iteration 27, loss = 0.09577196
Iteration 28, loss = 0.09252976
Iteration 29, loss = 0.08936770
Iteration 30, loss = 0.08630049


Kfold

In [65]:
Kfold = 5
skf = StratifiedKFold(n_splits=Kfold, shuffle=True, random_state=1)
skfind = [None] * Kfold

cont = 0
for index in skf.split(X, y):
    skfind[cont] = index
    cont += 1

print(f'Número de folds: {len(skfind)}')

Número de folds: 5


## Classificação

Kfold

In [ ]:
Kfold = 3
skf = StratifiedKFold(n_splits=Kfold, shuffle=True,random_state=42) 

Arquivo de métricas

In [25]:
def writeMetricFiles(file_name, metrics, best_metrics, final_metrics, stds, times, n=Kfold):
    with open(file_name, 'w') as f:
        f.write('Final Metrics:\n')
        f.write(f'Acc: {final_metrics[0]*100:.2f}%\n')
        f.write(f'Std Acc: {stds[0]*100:.2f}%\n')
        f.write(f'F1_score: {final_metrics[1]*100:.2f}%\n')
        f.write(f'Std F1_score: {stds[1]*100:.2f}%\n')
        f.write(f'Recall: {final_metrics[2]*100:.2f}%\n')
        f.write(f'Std Recall: {stds[2]*100:.2f}%\n')
        f.write(f'Precision: {final_metrics[3]*100:.2f}%\n')
        f.write(f'Std Precision: {stds[3]*100:.2f}%\n')

        f.write('--------------------- Best Results ---------------------\n')
        f.write(f'Acc: {best_metrics[0]*100:.2f}%\n')
        f.write(f'F1_score: {best_metrics[1]*100:.2f}%\n')
        f.write(f'Recall: {best_metrics[2]*100:.2f}%\n')
        f.write(f'Precision: {best_metrics[3]*100:.2f}%\n')
        f.write(f'Best Matrix:\n{best_metrics[4]}\n')

        f.write('--------------------- 5 iterations ---------------------\n')
        f.write(f'Acc: {metrics[0]}\n')
        f.write(f'F1_score: {metrics[1]}\n')
        f.write(f'Recall: {metrics[2]}\n')
        f.write(f'Precision: {metrics[3]}\n')
        f.write(f'Matrices: {metrics[4]}\n')

        f.write('--------------------- Times ---------------------\n')
        f.write(f'Train Time: {sum(times[0])/n:.4f}s\n')
        f.write(f'Test Time: {sum(times[1])/n:.4f}s\n')
    print('Arquivo gravado...')


PCA

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

def apply_pca(X_train, X_test):                                   -------- 10 (teste)    90 (treino)-> train test split fit_pca(9) (9 9 9 9 9 9 9 9 9 ) scaler e transforms
    scaler = StandardScaler()                                      
    X_train_scaled = scaler.fit_transform(X_train)                 
    X_test_scaled = scaler.transform(X_test)

    pca = PCA(n_components=0.95)

    X_train_pca = pca.fit_transform(X_train_scaled)
    X_test_pca = pca.transform(X_test_scaled)

    print("Shape reduzido do treino:", X_train_pca.shape) 
    print("Shape reduzido do teste:", X_test_pca.shape) 

    return X_train_pca, X_test_pca, scaler, pca

Modelos

In [ ]:
# Parâmetros pro gridsearch
model_params = {
    'knn': {
        'model': KNeighborsClassifier(),
        'params': {
            'n_neighbors': [3, 5, 7],
            'weights': ['uniform', 'distance'],
            'metric': ['euclidean', 'manhattan']
        }
    },
    'mlp': {
        'model': MLPClassifier(),
        'params': {
            'hidden_layer_sizes': [(50,), (100,), (50, 50)],
            'activation': ['relu', 'tanh'],
            'solver': ['adam'],
            'alpha': [0.0001, 0.001],
            'learning_rate': ['constant', 'adaptive']
    }
    },
    'svm': {
        'model': SVC(),
        'params': {
            'C': [0.1, 1, 10, 100, 1000],
            'kernel': ['rbf'],
        }
    }
}

rn_params = {
        'learning_rate': [1e-3, 1e-4],
        'batch_size': [32, 64],
        'epochs': [10, 20]
}

Classificadores

In [27]:
def SVM_classifier(X, y, skf, use_pca=False):
    conf_mat = np.zeros((2, 2))
    acc, prec, f1s, rec = [], [], [], []
    conf_matrixs = []

    acc_best = prec_best = f1_best = rec_best = -99
    matrix_best = None

    train_time, test_time = [], []

    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
        print(f'------------------------------- Rodada {fold} -------------------------------')

        X_train_raw, X_test_raw = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        if use_pca:
            X_train, X_test, scaler, pca = apply_pca(X_train_raw, X_test_raw)
        else:
            X_train, X_test = X_train_raw, X_test_raw

        print("Sizes:")
        print(f"X_train:{X_train.shape}\ty_train:{y_train.shape}")
        print(f"X_test:{X_test.shape}\ty_test:{y_test.shape}")

        print('Treinando....')
        svm_model = SVC(kernel='rbf')

        train_begin = time.time()
        svm_model.fit(X_train, y_train)
        train_time.append(time.time() - train_begin)

        print('========================================')
        print('Testando....')
        test_begin = time.time()
        y_pred = svm_model.predict(X_test)
        test_time.append(time.time() - test_begin)

        test_acc = accuracy_score(y_test, y_pred)
        print(f'{fold} Acurácia do teste: {test_acc}')

        cm = confusion_matrix(y_test, y_pred)
        conf_mat += cm
        print('------- Matriz de confusão --------')
        print(cm)

        # Métricas com average (ajuste se for multiclasse)
        acc.append(test_acc)
        prec.append(precision_score(y_test, y_pred, average='binary'))
        f1s.append(f1_score(y_test, y_pred, average='binary'))
        rec.append(recall_score(y_test, y_pred, average='binary'))
        conf_matrixs.append(cm)

        if test_acc > acc_best:
            acc_best = test_acc
            prec_best = precision_score(y_test, y_pred, average='binary')
            f1_best = f1_score(y_test, y_pred, average='binary')
            rec_best = recall_score(y_test, y_pred, average='binary')
            matrix_best = cm

        print('========================================\n')

    # Finais
    Kfold = len(acc)
    stds = [np.std(acc), np.std(f1s), np.std(rec), np.std(prec)]
    final_metrics = [np.mean(acc), np.mean(f1s), np.mean(rec), np.mean(prec), conf_mat]
    best_metrics = [acc_best, f1_best, rec_best, prec_best, matrix_best]
    metrics = [acc, f1s, rec, prec, conf_matrixs]
    times = [train_time, test_time]

    return metrics, best_metrics, final_metrics, stds, times


In [44]:
def KNN_classifier(X, y,skf, use_pca=False):
    conf_mat = np.zeros((2,2))
    acc = []
    prec = []
    f1s = []
    rec = []
    conf_matrixs = []

    acc_best = -99
    prec_best = -99
    f1_best = -99
    rec_best = -99
    matrix_best = -99

    train_time = []
    test_time = []

    test_total_size = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
      print(f'------------------------------- Rodada {fold} -------------------------------')
      
      X_train_raw, X_test_raw = X[train_idx], X[test_idx]
      y_train, y_test = y[train_idx], y[test_idx]

      if use_pca:
        X_train, X_test, scaler, pca = apply_pca(X_train_raw, X_test_raw)
      else:
        X_train, X_test = X_train_raw, X_test_raw

      print("Sizes:")
      print(f"X_train:{X_train.shape}\ty_train:{y_train.shape}")
      print(f"X_test:{X_test.shape}\ty_test:{y_test.shape}")

      print('Treinando....', end='\n')

      knn_model = KNeighborsClassifier(n_neighbors=5)

      train_begin = time.time()
      knn_model.fit(X_train, y_train)
      train_end = time.time()

      train_time.append((train_end - train_begin))

      print('========================================')
      print('Testando....', end='\n')

      test_begin = time.time()
      y_pred = knn_model.predict(X_test)
      test_end = time.time()

      test_time.append((test_end - test_begin))

      test_acc = accuracy_score(y_test, y_pred)
      print(f'{fold} Acurácia do teste: {test_acc}')

      cm = confusion_matrix(y_test, y_pred)
      conf_mat += cm

      print('------- Matriz de confusão --------')
      print(cm, end='\n')

      #Calcula as métricas da iteração
      acc.append(accuracy_score(y_test, y_pred))
      prec.append(precision_score(y_test, y_pred,average='binary'))
      f1s.append(f1_score(y_test, y_pred,average='binary'))
      rec.append(recall_score(y_test, y_pred,average='binary'))

      conf_matrixs.append(cm)

      #Calcula as melhores métricas
      if test_acc > acc_best:
        acc_best = test_acc
        prec_best = precision_score(y_test, y_pred)
        f1_best = f1_score(y_test, y_pred)
        rec_best = recall_score(y_test, y_pred)

        matrix_best = cm

      print('========================================')
      print()

    #Calcula os desvios padrões de todas as rodadas
    std_acc = np.std(acc)
    std_prec = np.std(prec)
    std_f1s = np.std(f1s)
    std_rec = np.std(rec)

    acc_final = sum(acc)/Kfold
    prec_final = sum(prec)/Kfold
    f1s_final = sum(f1s)/Kfold
    rec_final = sum(rec)/Kfold

    metrics = [acc, f1s, rec, prec, conf_matrixs]
    best_metrics = [acc_best, f1_best, rec_best, prec_best, matrix_best]
    final_metrics = [acc_final, f1s_final, rec_final, prec_final, conf_mat]
    stds = [std_acc, std_f1s, std_rec, std_prec]
    times = [train_time, test_time]


    return metrics, best_metrics, final_metrics, stds, times

In [48]:
def MLP_classifier(X, y,skf, use_pca=False):
    conf_mat = np.zeros((2,2))
    acc = []
    prec = []
    f1s = []
    rec = []
    conf_matrixs = []

    acc_best = -99
    prec_best = -99
    f1_best = -99
    rec_best = -99
    matrix_best = -99

    train_time = []
    test_time = []

    test_total_size = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
        print(f'------------------------------- Rodada {fold} -------------------------------')
        
        X_train_raw, X_test_raw = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        if use_pca:
            X_train, X_test, scaler, pca = apply_pca(X_train_raw, X_test_raw)
        else:
            X_train, X_test = X_train_raw, X_test_raw

        print("Sizes:")
        print(f"X_train:{X_train.shape}\ty_train:{y_train.shape}")
        print(f"X_test:{X_test.shape}\ty_test:{y_test.shape}")

        print('Treinando....', end='\n')

        mlp_model = MLPClassifier(random_state=1, max_iter = 300)

        train_begin = time.time()
        mlp_model.fit(X_train, y_train)
        train_end = time.time()

        train_time.append((train_end - train_begin))

        print('========================================')
        print('Testando....', end='\n')

        test_begin = time.time()
        y_pred = mlp_model.predict(X_test)
        test_end = time.time()

        test_time.append((test_end - test_begin))

        test_acc = accuracy_score(y_test, y_pred)
        print(f'{fold} Acurácia do teste: {test_acc}')

        cm = confusion_matrix(y_test, y_pred)
        conf_mat += cm

        print('------- Matriz de confusão --------')
        print(cm, end='\n')

        #Calcula as métricas da iteração
        acc.append(accuracy_score(y_test, y_pred))
        prec.append(precision_score(y_test, y_pred,average='binary'))
        f1s.append(f1_score(y_test, y_pred,average='binary'))
        rec.append(recall_score(y_test, y_pred,average='binary'))

        conf_matrixs.append(cm)

        #Calcula as melhores métricas
        if test_acc > acc_best:
            acc_best = test_acc
            prec_best = precision_score(y_test, y_pred)
            f1_best = f1_score(y_test, y_pred)
            rec_best = recall_score(y_test, y_pred)

            matrix_best = cm

        print('========================================')
        print()

    #Calcula os desvios padrões de todas as rodadas
    std_acc = np.std(acc)
    std_prec = np.std(prec)
    std_f1s = np.std(f1s)
    std_rec = np.std(rec)

    acc_final = sum(acc)/Kfold
    prec_final = sum(prec)/Kfold
    f1s_final = sum(f1s)/Kfold
    rec_final = sum(rec)/Kfold

    metrics = [acc, f1s, rec, prec, conf_matrixs]
    best_metrics = [acc_best, f1_best, rec_best, prec_best, matrix_best]
    final_metrics = [acc_final, f1s_final, rec_final, prec_final, conf_mat]
    stds = [std_acc, std_f1s, std_rec, std_prec]
    times = [train_time, test_time]


    return metrics, best_metrics, final_metrics, stds, times

In [49]:
def RF_classifier(X, y,skf, use_pca=False):
    conf_mat = np.zeros((2, 2))
    acc = []
    prec = []
    f1s = []
    rec = []
    conf_matrixs = []

    acc_best = -99
    prec_best = -99
    f1_best = -99
    rec_best = -99
    matrix_best = -99

    train_time = []
    test_time = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
        print(f'------------------------------- Rodada {fold} -------------------------------')
        X_train_raw, X_test_raw = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        if use_pca:
            X_train, X_test, scaler, pca = apply_pca(X_train_raw, X_test_raw)
        else:
            X_train, X_test = X_train_raw, X_test_raw

        print("Sizes:")
        print(f"X_train:{X_train.shape}\ty_train:{y_train.shape}")
        print(f"X_test:{X_test.shape}\ty_test:{y_test.shape}")

        print('Treinando....', end='\n')
        rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

        train_begin = time.time()
        rf_model.fit(X_train, y_train)
        train_end = time.time()
        train_time.append((train_end - train_begin))

        print('========================================')
        print('Testando....', end='\n')
        test_begin = time.time()
        y_pred = rf_model.predict(X_test)
        test_end = time.time()
        test_time.append((test_end - test_begin))

        test_acc = accuracy_score(y_test, y_pred)
        print(f'{fold} Acurácia do teste: {test_acc}')

        cm = confusion_matrix(y_test, y_pred)
        conf_mat += cm
        print('------- Matriz de confusão --------')
        print(cm)

        # Métricas
        acc.append(test_acc)
        prec.append(precision_score(y_test, y_pred, average='binary'))
        f1s.append(f1_score(y_test, y_pred, average='binary'))
        rec.append(recall_score(y_test, y_pred, average='binary'))

        conf_matrixs.append(cm)

        if test_acc > acc_best:
            acc_best = test_acc
            prec_best = precision_score(y_test, y_pred, average='binary')
            f1_best = f1_score(y_test, y_pred, average='binary')
            rec_best = recall_score(y_test, y_pred, average='binary')
            matrix_best = cm

        print('========================================\n')

    # Estatísticas finais
    std_acc = np.std(acc)
    std_prec = np.std(prec)
    std_f1s = np.std(f1s)
    std_rec = np.std(rec)

    acc_final = np.mean(acc)
    prec_final = np.mean(prec)
    f1s_final = np.mean(f1s)
    rec_final = np.mean(rec)

    metrics = [acc, f1s, rec, prec, conf_matrixs]
    best_metrics = [acc_best, f1_best, rec_best, prec_best, matrix_best]
    final_metrics = [acc_final, f1s_final, rec_final, prec_final, conf_mat]
    stds = [std_acc, std_f1s, std_rec, std_prec]
    times = [train_time, test_time]

    return metrics, best_metrics, final_metrics, stds, times


In [47]:
path_metrics=f'/home/labian-2/Desktop/Crateus/odonto_forense/logs binary/{fold_name}/ml_metrics/'

Sem PCA

In [ ]:
cnn_file_name = 'metricas_svm_sem_pca_' + fold_name + '.txt'  # nome do arquivo para salvar as métricas

# Chamada da função para treinar e avaliar o classificador MLP com k-fold
metrics, best_metrics, final_metrics, stds, times = SVM_classifier(X, y,skf)

# Grava as métricas no arquivo de texto
writeMetricFiles(path_metrics + cnn_file_name, metrics, best_metrics, final_metrics, stds, times)

In [ ]:
cnn_file_name = 'metricas_knn_sem_pca_' + fold_name + '.txt'  # nome do arquivo para salvar as métricas

# Chamada da função para treinar e avaliar o classificador MLP com k-fold
metrics, best_metrics, final_metrics, stds, times = KNN_classifier(X, y,skf)

# Grava as métricas no arquivo de texto
writeMetricFiles(path_metrics + cnn_file_name, metrics, best_metrics, final_metrics, stds, times)

------------------------------- Rodada 0 -------------------------------
Sizes:
X_train:(72, 512)	y_train:(72,)
X_test:(36, 512)	y_test:(36,)
Treinando....
Testando....
0 Acurácia do teste: 1.0
------- Matriz de confusão --------
[[18  0]
 [ 0 18]]

------------------------------- Rodada 1 -------------------------------
Sizes:
X_train:(72, 512)	y_train:(72,)
X_test:(36, 512)	y_test:(36,)
Treinando....
Testando....
1 Acurácia do teste: 0.9722222222222222
------- Matriz de confusão --------
[[17  1]
 [ 0 18]]

------------------------------- Rodada 2 -------------------------------
Sizes:
X_train:(72, 512)	y_train:(72,)
X_test:(36, 512)	y_test:(36,)
Treinando....
Testando....
2 Acurácia do teste: 0.8611111111111112
------- Matriz de confusão --------
[[16  2]
 [ 3 15]]

Arquivo gravado...


In [50]:
cnn_file_name = 'metricas_mlp_sem_pca_' + fold_name + '.txt'  # nome do arquivo para salvar as métricas

# Chamada da função para treinar e avaliar o classificador MLP com k-fold
metrics, best_metrics, final_metrics, stds, times = MLP_classifier(X, y,skf)

# Grava as métricas no arquivo de texto
writeMetricFiles(path_metrics + cnn_file_name, metrics, best_metrics, final_metrics, stds, times)

------------------------------- Rodada 0 -------------------------------
Sizes:
X_train:(72, 512)	y_train:(72,)
X_test:(36, 512)	y_test:(36,)
Treinando....
Testando....
0 Acurácia do teste: 1.0
------- Matriz de confusão --------
[[18  0]
 [ 0 18]]

------------------------------- Rodada 1 -------------------------------
Sizes:
X_train:(72, 512)	y_train:(72,)
X_test:(36, 512)	y_test:(36,)
Treinando....
Testando....
1 Acurácia do teste: 0.9444444444444444
------- Matriz de confusão --------
[[16  2]
 [ 0 18]]

------------------------------- Rodada 2 -------------------------------
Sizes:
X_train:(72, 512)	y_train:(72,)
X_test:(36, 512)	y_test:(36,)
Treinando....
Testando....
2 Acurácia do teste: 0.8611111111111112
------- Matriz de confusão --------
[[17  1]
 [ 4 14]]

Arquivo gravado...


In [52]:
cnn_file_name = 'metricas_rf_sem_pca_' + fold_name + '.txt'  # nome do arquivo para salvar as métricas

# Chamada da função para treinar e avaliar o classificador MLP com k-fold
metrics, best_metrics, final_metrics, stds, times = RF_classifier(X, y,skf)

# Grava as métricas no arquivo de texto
writeMetricFiles(path_metrics + cnn_file_name, metrics, best_metrics, final_metrics, stds, times)

------------------------------- Rodada 0 -------------------------------
Sizes:
X_train:(72, 512)	y_train:(72,)
X_test:(36, 512)	y_test:(36,)
Treinando....
Testando....
0 Acurácia do teste: 1.0
------- Matriz de confusão --------
[[18  0]
 [ 0 18]]

------------------------------- Rodada 1 -------------------------------
Sizes:
X_train:(72, 512)	y_train:(72,)
X_test:(36, 512)	y_test:(36,)
Treinando....
Testando....
1 Acurácia do teste: 0.9166666666666666
------- Matriz de confusão --------
[[15  3]
 [ 0 18]]

------------------------------- Rodada 2 -------------------------------
Sizes:
X_train:(72, 512)	y_train:(72,)
X_test:(36, 512)	y_test:(36,)
Treinando....
Testando....
2 Acurácia do teste: 0.8888888888888888
------- Matriz de confusão --------
[[17  1]
 [ 3 15]]

Arquivo gravado...


Com PCA

In [32]:
path_metrics=f'/home/labian-2/Desktop/Crateus/odonto_forense/logs binary/{fold_name}/ml_metrics/'

cnn_file_name = 'metricas_svm_com_pca_' + fold_name + '.txt'  # nome do arquivo para salvar as métricas

# Chamada da função para treinar e avaliar o classificador MLP com k-fold
metrics, best_metrics, final_metrics, stds, times = SVM_classifier(X, y,skf,use_pca=True)

# Grava as métricas no arquivo de texto
writeMetricFiles(path_metrics + cnn_file_name, metrics, best_metrics, final_metrics, stds, times)

------------------------------- Rodada 0 -------------------------------
Shape reduzido do treino: (72, 26)
Shape reduzido do teste: (36, 26)
Sizes:
X_train:(72, 26)	y_train:(72,)
X_test:(36, 26)	y_test:(36,)
Treinando....
Testando....
0 Acurácia do teste: 1.0
------- Matriz de confusão --------
[[18  0]
 [ 0 18]]

------------------------------- Rodada 1 -------------------------------
Shape reduzido do treino: (72, 27)
Shape reduzido do teste: (36, 27)
Sizes:
X_train:(72, 27)	y_train:(72,)
X_test:(36, 27)	y_test:(36,)
Treinando....
Testando....
1 Acurácia do teste: 0.9722222222222222
------- Matriz de confusão --------
[[17  1]
 [ 0 18]]

------------------------------- Rodada 2 -------------------------------
Shape reduzido do treino: (72, 24)
Shape reduzido do teste: (36, 24)
Sizes:
X_train:(72, 24)	y_train:(72,)
X_test:(36, 24)	y_test:(36,)
Treinando....
Testando....
2 Acurácia do teste: 0.8611111111111112
------- Matriz de confusão --------
[[16  2]
 [ 3 15]]

Arquivo gravado...


In [46]:
path_metrics=f'/home/labian-2/Desktop/Crateus/odonto_forense/logs binary/{fold_name}/ml_metrics/'

cnn_file_name = 'metricas_knn_com_pca_' + fold_name + '.txt'  # nome do arquivo para salvar as métricas

# Chamada da função para treinar e avaliar o classificador MLP com k-fold
metrics, best_metrics, final_metrics, stds, times = KNN_classifier(X, y,skf,use_pca=True)

# Grava as métricas no arquivo de texto
writeMetricFiles(path_metrics + cnn_file_name, metrics, best_metrics, final_metrics, stds, times)

------------------------------- Rodada 0 -------------------------------
Shape reduzido do treino: (72, 26)
Shape reduzido do teste: (36, 26)
Sizes:
X_train:(72, 26)	y_train:(72,)
X_test:(36, 26)	y_test:(36,)
Treinando....
Testando....
0 Acurácia do teste: 1.0
------- Matriz de confusão --------
[[18  0]
 [ 0 18]]

------------------------------- Rodada 1 -------------------------------
Shape reduzido do treino: (72, 27)
Shape reduzido do teste: (36, 27)
Sizes:
X_train:(72, 27)	y_train:(72,)
X_test:(36, 27)	y_test:(36,)
Treinando....
Testando....
1 Acurácia do teste: 1.0
------- Matriz de confusão --------
[[18  0]
 [ 0 18]]

------------------------------- Rodada 2 -------------------------------
Shape reduzido do treino: (72, 24)
Shape reduzido do teste: (36, 24)
Sizes:
X_train:(72, 24)	y_train:(72,)
X_test:(36, 24)	y_test:(36,)
Treinando....
Testando....
2 Acurácia do teste: 0.8611111111111112
------- Matriz de confusão --------
[[16  2]
 [ 3 15]]

Arquivo gravado...


In [51]:
cnn_file_name = 'metricas_mlp_com_pca_' + fold_name + '.txt'  # nome do arquivo para salvar as métricas

# Chamada da função para treinar e avaliar o classificador MLP com k-fold
metrics, best_metrics, final_metrics, stds, times = MLP_classifier(X, y,skf,use_pca=True)

# Grava as métricas no arquivo de texto
writeMetricFiles(path_metrics + cnn_file_name, metrics, best_metrics, final_metrics, stds, times)

------------------------------- Rodada 0 -------------------------------
Shape reduzido do treino: (72, 26)
Shape reduzido do teste: (36, 26)
Sizes:
X_train:(72, 26)	y_train:(72,)
X_test:(36, 26)	y_test:(36,)
Treinando....
Testando....
0 Acurácia do teste: 0.9722222222222222
------- Matriz de confusão --------
[[18  0]
 [ 1 17]]

------------------------------- Rodada 1 -------------------------------
Shape reduzido do treino: (72, 27)
Shape reduzido do teste: (36, 27)
Sizes:
X_train:(72, 27)	y_train:(72,)
X_test:(36, 27)	y_test:(36,)
Treinando....
Testando....
1 Acurácia do teste: 0.9444444444444444
------- Matriz de confusão --------
[[16  2]
 [ 0 18]]

------------------------------- Rodada 2 -------------------------------
Shape reduzido do treino: (72, 24)
Shape reduzido do teste: (36, 24)
Sizes:
X_train:(72, 24)	y_train:(72,)
X_test:(36, 24)	y_test:(36,)
Treinando....
Testando....
2 Acurácia do teste: 0.8611111111111112
------- Matriz de confusão --------
[[17  1]
 [ 4 14]]

Arqu

In [53]:
cnn_file_name = 'metricas_rf_com_pca_' + fold_name + '.txt'  # nome do arquivo para salvar as métricas

# Chamada da função para treinar e avaliar o classificador MLP com k-fold
metrics, best_metrics, final_metrics, stds, times = RF_classifier(X, y,skf,use_pca=True)

# Grava as métricas no arquivo de texto
writeMetricFiles(path_metrics + cnn_file_name, metrics, best_metrics, final_metrics, stds, times)

------------------------------- Rodada 0 -------------------------------
Shape reduzido do treino: (72, 26)
Shape reduzido do teste: (36, 26)
Sizes:
X_train:(72, 26)	y_train:(72,)
X_test:(36, 26)	y_test:(36,)
Treinando....
Testando....
0 Acurácia do teste: 1.0
------- Matriz de confusão --------
[[18  0]
 [ 0 18]]

------------------------------- Rodada 1 -------------------------------
Shape reduzido do treino: (72, 27)
Shape reduzido do teste: (36, 27)
Sizes:
X_train:(72, 27)	y_train:(72,)
X_test:(36, 27)	y_test:(36,)
Treinando....
Testando....
1 Acurácia do teste: 1.0
------- Matriz de confusão --------
[[18  0]
 [ 0 18]]

------------------------------- Rodada 2 -------------------------------
Shape reduzido do treino: (72, 24)
Shape reduzido do teste: (36, 24)
Sizes:
X_train:(72, 24)	y_train:(72,)
X_test:(36, 24)	y_test:(36,)
Treinando....
Testando....
2 Acurácia do teste: 0.8888888888888888
------- Matriz de confusão --------
[[17  1]
 [ 3 15]]

Arquivo gravado...
